# Statistical Arbitrage on Crypto, Backtested
Pairs trading, the oldest statistical arbitrage: find two coins whose prices move together, and when
they drift apart, buy the cheap one and short the expensive one until they meet again. The rules are
the ones Gatev, Goetzmann and Rouwenhorst tested on US stocks (Review of Financial Studies, 2006).
Ten Binance perpetuals, 2021 to 2026, with fees and funding. Education, not financial advice.

The rules: over 12 months (formation), every coin's price is scaled to start at 1; the pairs with the
smallest sum of squared differences are the closest. Over the next 6 months (trading), open when the
two scaled prices are more than 2 standard deviations apart — the standard deviation measured in the
formation months — and close when they cross, or when the 6 months end.

## 1. Install
freqtrade is the backtester; statsmodels has the cointegration test we use in section 13.

In [ ]:
!pip install -q freqtrade==2026.10 statsmodels

## 2. Helpers from the earlier episodes
`binance_vision` — raw Binance data with SHA-256 hash checks (episode q00); `quant_report` — the verdict and hourly marked-to-market equity (episode q01).

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py -O https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/quant_report.py
from binance_vision import save_for_freqtrade, DATA
from quant_report import verdict, equity, drawdown
import itertools
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. The coins
The ten perpetuals of episode q03 that Binance still lists (the coin EOS was delisted in 2025). Daily closes from 2020; a missing day keeps the last price. Uses `save_for_freqtrade`, `DATA` (section 2).

In [ ]:
coins = ["BTC", "ETH", "BCH", "XRP", "LTC",
         "TRX", "ETC", "LINK", "XLM", "ADA"]
for c in coins:
    save_for_freqtrade(c + "USDT", "1d", "2020-01", "2026-10")
daily = str(DATA / "{}_USDT_USDT-1d-futures.feather")
close = pd.DataFrame({c: pd.read_feather(daily.format(c))
                         .set_index("date")["close"] for c in coins})
close = close.ffill()                   # a missing day keeps the last price
print(close.loc["2020-02-01":].notna().sum().to_string())    # days per coin

## 4. How close are two coins?
Scale each price to start at 1, then add up the squared gaps day by day: the sum of squared deviations, $SSD = \sum_t (p^A_t - p^B_t)^2$. Smallest = closest. Ten coins make $10 \cdot 9 / 2 = 45$ pairs. Uses `close` (section 3).

In [ ]:
def distances(prices):
    norm = prices / prices.iloc[0]                  # every coin starts at 1
    rows = [(a, b, ((norm[a] - norm[b]) ** 2).sum())
            for a, b in itertools.combinations(prices.columns, 2)]
    return pd.DataFrame(rows, columns=["a", "b", "ssd"]).sort_values("ssd")

first = close.loc["2020-02-01":"2021-01-31"]     # the first 12 months
d = distances(first)
print(len(d), "pairs; the closest six:")
print(d.head(6).round(1).to_string(index=False))

## 5. Three pairs every six months
Formation = the 12 months before each trading period. A coin can sit in only one pair, because freqtrade holds one position per coin. Uses `close` (3), `distances` (4).

In [ ]:
def pick(prices, n=3):
    chosen, used = [], set()
    for a, b, ssd in distances(prices).itertuples(index=False):
        if a not in used and b not in used:          # one coin, one pair
            chosen.append((a, b))
            used |= {a, b}
        if len(chosen) == n:
            return chosen

def formation(start):
    return close.loc[start - pd.DateOffset(months=12):
                     start - pd.Timedelta("1D")]

windows = []
for start in pd.date_range("2021-02-01", "2026-08-01", freq="6MS", tz="UTC"):
    end = min(start + pd.DateOffset(months=6) - pd.Timedelta("1D"),
              close.index[-1])
    windows.append((start, end, pick(formation(start))))
    print(start.date(), "to", end.date(), windows[-1][2])

## 6. The spread, and when to open
Spread = scaled price of A − scaled price of B. In the trading months both start at 1 again on the last formation day; the trigger is 2 standard deviations of the spread **in the formation months**. Uses `close` (3), `formation`, `windows` (5).

In [ ]:
def spread(prices, a, b):
    norm = prices / prices.iloc[0]
    return norm[a] - norm[b]

def trading(start, end):            # from the last formation close on
    return close.loc[start - pd.Timedelta("1D"):end]

start, end, pairs = windows[0]
a, b = pairs[0]
trigger = 2 * spread(formation(start), a, b).std()
s = spread(trading(start, end), a, b)
print(f"{a}/{b}: trigger {trigger:.3f}, first day beyond it:",
      s[s.abs() > trigger].index[0].date())

### On a chart
Top: the two scaled prices. Bottom: the spread and the ±2 standard deviation trigger. Uses `s`, `trigger`, `a`, `b`, `start`, `end`, `trading` (above).

In [ ]:
fig, (top, low) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
norm = trading(start, end)[[a, b]] / trading(start, end)[[a, b]].iloc[0]
top.plot(norm.index, norm[a], label=a)
top.plot(norm.index, norm[b], label=b)
top.legend(); top.set_title("scaled prices: both start at 1")
low.plot(s.index, s, color="gray", label="spread")
for level in (trigger, -trigger):
    low.axhline(level, color="red", ls="--")
low.axhline(0, color="black", lw=0.8)
low.legend(); low.set_title("spread and the 2 standard deviation trigger")
plt.show()

## 7. From the spread to positions
+1 = long A and short B, −1 = the opposite, 0 = flat. Open beyond the trigger, close when the spread changes sign (the prices crossed); on the last day everything closes. Uses `spread`, `trading`, `formation` (5–6).

In [ ]:
def positions(s, trigger):
    pos, out = 0, []
    for today, before in zip(s, s.shift()):
        if pos == 0 and abs(today) > trigger:
            pos = -1 if today > 0 else 1        # short the expensive one
        elif pos != 0 and today * before <= 0:  # the prices crossed
            pos = 0
        out.append(pos)
    out[-1] = 0                                 # the 6 months are over
    return pd.Series(out, s.index)

def distance_rule(a, b, start, end):
    trigger = 2 * spread(formation(start), a, b).std()
    s = spread(trading(start, end), a, b)
    return positions(s, trigger).loc[start:]

## 8. One table for freqtrade
A column per coin, a row per day: +1 long, −1 short, 0 flat. Leg B always takes the opposite side of leg A. Uses `windows` (5), `distance_rule` (7), `close`, `coins` (3).

In [ ]:
def signals(windows, rule, name):
    sig = pd.DataFrame(0, index=close.index, columns=coins)
    for start, end, pairs in windows:
        for a, b in pairs:
            p = rule(a, b, start, end)
            sig.loc[p.index, a] = p
            sig.loc[p.index, b] = -p                   # the other leg
    sig.to_csv(name)
    opened = ((sig != 0) & (sig.shift() == 0)).sum().sum() // 2
    print(name, "-", opened, "pair trades opened")
    return sig

sig = signals(windows, distance_rule, "signals_distance.csv")

## 9. Backtest settings
Isolated margin, leverage 1, six slots — two legs for each of three pairs. Every day is checked against hourly candles, so a liquidation inside the day is seen. 10,000 USDT (Tether, a dollar stablecoin). Uses `coins` (section 3).

In [ ]:
config = {
    "trading_mode": "futures",
    "margin_mode": "isolated",               # each position has its own margin
    "liquidation_buffer": 0.0,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",             # account / free slots
    "dry_run_wallet": 10000, "dry_run": True,
    "max_open_trades": 6,                    # 3 pairs x 2 legs
    "timeframe": "1d", "timeframe_detail": "1h", "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True,
                      "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True,
                     "order_book_top": 1},
    "exchange": {"name": "binance",
                 "pair_whitelist": [c + "/USDT:USDT" for c in coins]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 10. Strategy file, part 1: the settings
No profit target and no stop: a leg closes when its pair closes — or when the exchange liquidates it (part 3).

In [ ]:
%%writefile user_data/strategies/Pairs.py
import pandas as pd
from freqtrade.strategy import IStrategy, stoploss_from_absolute

class Pairs(IStrategy):
    SIGNALS = "signals_distance.csv"     # written in section 8
    MMR = 0.004                          # maintenance margin rate, q00
    timeframe = "1d"
    can_short = True
    minimal_roi = {"0": 100}             # no profit target
    stoploss = -0.99                     # no stop of our own
    use_custom_stoploss = True

## 11. Part 2: read the table, enter and exit
Each coin reads its own column. Enter only on the day the column changes — a liquidated leg is not bought back. Signals at the daily close, orders at the next open. `SIGNALS` from section 10.

In [ ]:
%%writefile -a user_data/strategies/Pairs.py

    def populate_indicators(self, df, metadata):
        coin = metadata["pair"].split("/")[0]       # BTC/USDT:USDT -> BTC
        sig = pd.read_csv(self.SIGNALS, index_col=0, parse_dates=True)
        df["pos"] = df["date"].map(sig[coin]).fillna(0)
        return df

    def populate_entry_trend(self, df, metadata):
        new = df["pos"] != df["pos"].shift()        # the day it changes
        df.loc[new & (df["pos"] == 1), "enter_long"] = 1
        df.loc[new & (df["pos"] == -1), "enter_short"] = 1
        return df

    def populate_exit_trend(self, df, metadata):
        df.loc[df["pos"] != 1, "exit_long"] = 1
        df.loc[df["pos"] != -1, "exit_short"] = 1
        return df

## 12. Part 3: the liquidation price
The formula from episode q00, funding included: a short at leverage 1 is liquidated when the price about doubles. `MMR` from section 10.

In [ ]:
%%writefile -a user_data/strategies/Pairs.py

    def custom_stoploss(self, pair, trade, current_time, current_rate,
                        current_profit, after_fill, **kwargs):
        m = 1 + (trade.funding_fees or 0) / trade.stake_amount   # margin left
        if trade.is_short:
            liq = trade.open_rate * (1 + m / trade.leverage) / (1 + self.MMR)
        else:
            liq = trade.open_rate * (1 - m / trade.leverage) / (1 - self.MMR)
        return stoploss_from_absolute(liq, current_rate,
                                      is_short=trade.is_short,
                                      leverage=trade.leverage)

## 13. Run it
1 February 2021 to 31 October 2026. Uses `config.json` (9), `Pairs` (10–12), `verdict`, `equity`, `drawdown` (2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s Pairs --timerange 20210201-20261031 --export trades > run.log 2>&1
dist, trades = verdict("Pairs")
dist["drawdown, marked to market"] = drawdown(equity(trades,
                                                     start="2021-02-01"))

## 14. Which pairs made it, which lost it
Profit of each pair in each six months, and how its trades ended: the prices crossed, the six months ran out, or a leg was liquidated. Uses `trades` (13), `windows` (5).

In [ ]:
trades["coin"] = trades["pair"].str.split("/").str[0]
def ended(t, end):
    if "stop_loss" in t["exit_reason"]:       # our liquidation price, part 3
        return "liquidated"
    if t["close_date"].normalize() > end:     # closed the next morning
        return "time ran out"
    if t["exit_reason"] == "force_exit":      # the backtest itself ended
        return "time ran out"
    return "prices crossed"

rows = []
for start, end, pairs in windows:
    inside = trades[trades["open_date"].between(start, end)]
    for a, b in pairs:
        legs = inside[inside["coin"].isin([a, b])]
        for _, t in legs.iterrows():
            rows.append((start.date(), f"{a}/{b}", t["coin"],
                         "short" if t["is_short"] else "long",
                         ended(t, end), t["profit_abs"], t["funding_fees"]))
legs = pd.DataFrame(rows, columns=["start", "pair", "coin", "side", "ended",
                                   "profit", "funding"])
by_end = legs.groupby("ended")["profit"].agg(["size", "sum"])
print(by_end.round(0).to_string())                  # legs and their money
per_pair = legs.groupby(["start", "pair"])["profit"].sum()
for label, key in (("best", per_pair.idxmax()), ("worst", per_pair.idxmin())):
    print(label, "pair:", key[0], key[1], round(per_pair[key]))
    one = legs[(legs["start"] == key[0]) & (legs["pair"] == key[1])]
    print(one.drop(columns=["start", "pair"]).round(0).to_string(index=False))

### The liquidations, and who paid the funding
Every leg that hit its liquidation price (part 3), then funding by side, and the leg that paid the most. Uses `trades` (13, 14).

In [ ]:
liq = trades[trades["exit_reason"].str.contains("stop_loss")]
for _, t in liq.iterrows():
    print(f"{t['coin']:<4} short {t['open_rate']:<8.5g}"
          f" -> {t['close_rate']:<8.5g}"
          f" x{t['close_rate'] / t['open_rate']:.2f}  {t['close_date'].date()}"
          f"  {t['profit_abs']:,.0f}")
side = np.where(trades["is_short"], "shorts", "longs")
print(trades.groupby(side)["funding_fees"].sum().round(0).to_string())
t = trades.loc[trades["funding_fees"].idxmin()]        # paid the most
print("most funding:", t["coin"], "long" if not t["is_short"] else "short",
      t["open_date"].date(), "->", t["close_date"].date(),
      f"stake {t['stake_amount']:,.0f}, funding {t['funding_fees']:,.0f}")

## 15. The version most tutorials teach: cointegration
Engle–Granger test on the log prices of every pair; the three with the lowest p-value (the chance of a link this strong between two unrelated random walks). Spread = $\log A - \beta \log B$, $\beta$ from a straight-line fit; its z-score over 30 days; open beyond ±2, close when it crosses 0. Uses `close`, `coins` (3), `formation`, `windows`, `positions`, `signals` (5–8).

In [ ]:
from statsmodels.tsa.stattools import coint

def pick_coint(prices, n=3):
    logs = np.log(prices)
    tests = sorted((coint(logs[a], logs[b])[1], a, b)    # [1] = p-value
                   for a, b in itertools.combinations(coins, 2))
    chosen, used = [], set()
    for p, a, b in tests:
        if a not in used and b not in used:
            chosen.append((a, b))
            used |= {a, b}
        if len(chosen) == n:
            return chosen

def zscore_rule(a, b, start, end):
    logs = np.log(formation(start))
    beta = np.polyfit(logs[b], logs[a], 1)[0]           # A = beta * B + c
    recent = np.log(close.loc[start - pd.Timedelta("60D"):end])
    s = recent[a] - beta * recent[b]
    z = (s - s.rolling(30).mean()) / s.rolling(30).std()
    return positions(z.loc[start:], 2)

windows_c = [(start, end, pick_coint(formation(start)))
             for start, end, _ in windows]
sig_c = signals(windows_c, zscore_rule, "signals_coint.csv")

### Does a cointegrated pair stay cointegrated?
For every six months: the pairs that passed the test (p-value below 0.05) in the formation year — how many still pass over the next six months? By pure chance about 45 × 0.05 ≈ 2 of 45 pass. Uses `coins` (3), `windows` (5), `formation` (5), `trading` (6), `coint` (above).

In [ ]:
passed = kept = 0
for start, end, _ in windows:
    before, after = np.log(formation(start)), np.log(trading(start, end))
    for a, b in itertools.combinations(coins, 2):
        if coint(before[a], before[b])[1] < 0.05:
            passed += 1
            kept += coint(after[a], after[b])[1] < 0.05
print(f"passed in the formation year: {passed} "
      f"({passed / len(windows):.1f} of 45 per period)")
print(f"still passed in the next six months: {kept} ({kept / passed:.0%})")

## 16. The cheat: pairs picked with hindsight
The same distance rule, but the pairs and their standard deviation come from the whole of 2021–2026 — information nobody had in 2021. Uses `close` (3), `pick`, `windows`, `spread`, `trading`, `positions`, `signals` (5–8).

In [ ]:
everything = close.loc["2021-02-01":"2026-10-31"]
hindsight_pairs = pick(everything)
print("pairs picked with hindsight:", hindsight_pairs)

def hindsight_rule(a, b, start, end):
    trigger = 2 * spread(everything, a, b).std()       # from the future
    return positions(spread(trading(start, end), a, b),
                     trigger).loc[start:]

windows_h = [(start, end, hindsight_pairs) for start, end, _ in windows]
sig_h = signals(windows_h, hindsight_rule, "signals_hindsight.csv")

### Run both variants
Subclasses of `Pairs` (section 10) that read the other two tables. Uses `config.json` (9), `verdict`, `equity`, `drawdown` (2).

In [ ]:
%%writefile user_data/strategies/PairsVariants.py
from Pairs import Pairs

class PairsCoint(Pairs):
    SIGNALS = "signals_coint.csv"

class PairsHindsight(Pairs):
    SIGNALS = "signals_hindsight.csv"

In [ ]:
runs = {}
for name in ["PairsCoint", "PairsHindsight"]:
    !freqtrade backtesting --cache none -c config.json -s {name} --timerange 20210201-20261031 --export trades > run.log 2>&1
    runs[name], t = verdict(name)
    runs[name]["drawdown, marked to market"] = drawdown(
        equity(t, start="2021-02-01"))
    runs[name + " trades"] = t

## 17. Against holding the same coins
Buy-and-hold = the ten coins, equal amounts, rebalanced daily. Uses `close` (3), `trades` (13), `runs` (16), `equity`, `drawdown` (2).

In [ ]:
hold = 10_000 * (1 + close.loc["2021-02-01":].pct_change()
                 .mean(axis=1).fillna(0)).cumprod()
curves = {"buy and hold, 10 coins": hold,
          "distance (the paper)": equity(trades, start="2021-02-01"),
          "cointegration": equity(runs["PairsCoint trades"],
                                  start="2021-02-01"),
          "hindsight pairs": equity(runs["PairsHindsight trades"],
                                    start="2021-02-01")}
for name, curve in curves.items():
    print(f"{name:>22}: final {curve.iloc[-1]:>8.0f}"
          f"  max drawdown {drawdown(curve):.0%}")
plt.figure(figsize=(11, 5))
for name, curve in curves.items():
    plt.plot(curve.index, curve, label=name)
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2021 to 2026")
plt.show()

## 18. The Sharpe ratio: return per unit of wobble
The same four lines as episode q03: Sharpe ratio $S = \dfrac{\text{mean of the daily returns}}{\text{their standard deviation}} \times \sqrt{365}$, the hourly account (section 17) taken once a day; risk-free rate 0: idle USDT (Tether, a dollar stablecoin) earns nothing here. The paper's top 20 pairs: a monthly Sharpe of 0.59, so $0.59 \times \sqrt{12} \approx 2.0$ a year. Uses `curves` (17), `dist` (13), `runs` (16).

In [ ]:
def sharpe(curve):
    day = curve.resample("1D").last()           # the account once a day
    r = day.pct_change().dropna()               # daily returns
    return r.mean() / r.std() * 365 ** 0.5      # a year = 365 crypto days

for name, curve in curves.items():
    r = curve.resample("1D").last().pct_change().dropna()
    print(f"{name:>22}: mean {r.mean():.3%}  std {r.std():.2%}"
          f"  Sharpe {sharpe(curve):.2f}")
dist["Sharpe"] = sharpe(curves["distance (the paper)"])
runs["PairsCoint"]["Sharpe"] = sharpe(curves["cointegration"])
runs["PairsHindsight"]["Sharpe"] = sharpe(curves["hindsight pairs"])
paper = 0.59 * 12 ** 0.5                        # monthly -> a year
print(f"the paper, top 20 pairs: 0.59 x sqrt(12) = {paper:.2f}")

## 19. Every run side by side
A legend explains every row. Uses `dist` (13, 18), `runs` (16, 18).

In [ ]:
table = pd.DataFrame({"distance": dist, "cointegration": runs["PairsCoint"],
                      "hindsight": runs["PairsHindsight"]})
table.index = ["start", "final", "CAGR", "drawdown (freqtrade)", "trades",
               "long profit", "short profit", "funding", "fees",
               "drawdown (hourly)", "Sharpe"]
pct = ["CAGR", "drawdown (freqtrade)", "drawdown (hourly)"]
def cell(row, v):                       # how each row is written
    if row in pct:
        return f"{v:.0%}"
    return f"{v:.2f}" if row == "Sharpe" else f"{v:,.0f}"
shown_table = table.apply(lambda r: r.map(lambda v: cell(r.name, v)), axis=1)
print(shown_table.to_string())
print("money in USDT (Tether, a dollar stablecoin); trades = legs")
print("distance: the paper's rule; cointegration: the tutorial version;")
print("  hindsight: pairs picked knowing 2021-2026")
print("CAGR = compound annual growth rate: growth per year")
print("drawdown = deepest fall from a peak; freqtrade: closed trades only,")
print("  hourly: the whole account valued every hour")
print("funding = paid between longs and shorts every 8 hours")
print("fees = the exchange fee, 0.05% of every order")
print("Sharpe = mean daily return / its standard deviation x sqrt(365);")
print("  above 1 is the usual target; the paper's pairs: about 2")